# R5.4 - CropFusion Multimodal Training (replication-ready config)

Production training run on the frozen R5.2.7/R5.2.8 corpus with the R5.4
optimization stack (all committed under training/config and training/training):

- crop/macro_f1 over the full 5-class set is the monitor / early-stop metric
- restore_best_on_stop loads best.pt weights before the final evaluation
- warmup_cosine (2 warmup epochs) then cosine decay (scheduler stepping bug fixed)
- AdamW 1e-4 with backbone_lr_multiplier 0.3 (backbone ~3e-5, heads 1e-4)
- Staged backbone fine-tuning: stem stays frozen, blocks.6/.5 @10, blocks.4/.3 @20
- Geometric-only augmentation (no brightness/contrast jitter, no noise)
- sqrt_inv class weights (max ~3.1, no fabricated minority samples)

Frozen corpus: 10,674 observations; split: spatial leave-one-taluk-out
(Belthangady+Mangalore+Puttur train / Bantwal val / Sullia test).
Model: CropFusion multimodal (TabTransformer + EfficientNetV2-S + Temporal
Transformer + Cross Attention + Adaptive Gate).

Attach this Kaggle dataset:
- `shathanandabhatn/crop-yield-forecasting-karnataka-dakshina-kannada` (imagery)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/Brijesh2005/CropPrep.git'
REPO_ROOT = Path('/kaggle/working/CropPrep')

if not (REPO_ROOT / '.git').exists():
    print(f'cloning CropPrep -> {REPO_ROOT}')
    subprocess.run(
        ['git', 'clone', '--depth', '1', '-b', 'main', REPO_URL, str(REPO_ROOT)],
        check=True,
    )

sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print(f'repo root: {REPO_ROOT}')
print(f'cwd: {os.getcwd()}')

## 0.1 P100 GPU fix

Reinstall torch from cu126 index for Pascal P100 compatibility.

In [ ]:
import subprocess, sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchvision', 'torchaudio'],
    check=False,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
     'torch==2.10.0', 'torchvision==0.25.0', 'torchaudio==2.10.0',
     '--index-url', 'https://download.pytorch.org/whl/cu126'],
    check=True,
)
import torch
print('torch', torch.__version__,
      '| cuda', torch.cuda.is_available(),
      '| device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
      '| arch', torch.cuda.get_arch_list())

## 1. Environment verification

Verify GPU availability. STOP if unavailable.

In [ ]:
import torch, sys, platform

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA version:', torch.version.cuda)
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')
    print('Compute capability:', torch.cuda.get_device_capability())
else:
    raise RuntimeError('GPU REQUIRED FOR R5.4 TRAINING. STOP.')

## 2. Frozen Data Contract Gate

Verify the frozen corpus before training.

In [ ]:
import json, csv

manifest_path = REPO_ROOT / 'training_manifests' / 'crop_supervised_v1_manifest.json'
csv_path = REPO_ROOT / 'govt_crop_matched_v1' / 'crop_supervised_v1.csv'

manifest = json.loads(manifest_path.read_text(encoding='utf-8'))

with open(csv_path, newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

class_counts = {}
split_counts = {'train': 0, 'val': 0, 'test': 0}
TALUK_SPLIT = {'Belthangady': 'train', 'Mangalore': 'train', 'Puttur': 'train',
               'Bantwal': 'val', 'Sullia': 'test'}
for r in rows:
    c = r['crop_label']
    class_counts[c] = class_counts.get(c, 0) + 1
    s = TALUK_SPLIT.get(r['location_taluk'], 'unknown')
    split_counts[s] += 1

print('=' * 50)
print('  R5.4 FROZEN DATA CONTRACT')
print('=' * 50)
print(f'  Manifest: {manifest_path}')
print(f'  Dataset version: {manifest["dataset_version"]}')
print(f'  Total: {len(rows)}')
print(f'  Train: {split_counts["train"]}')
print(f'  Validation: {split_counts["val"]}')
print(f'  Test: {split_counts["test"]}')
print('  Classes:')
for c in sorted(class_counts.keys()):
    print(f'    {c}: {class_counts[c]}')
print('=' * 50)

assert len(rows) == 10674, f'Expected 10674, got {len(rows)}'
assert split_counts['train'] == 6116, f'Expected train=6116, got {split_counts["train"]}'
assert split_counts['val'] == 2267, f'Expected val=2267, got {split_counts["val"]}'
assert split_counts['test'] == 2291, f'Expected test=2291, got {split_counts["test"]}'
print('\nFROZEN DATA CONTRACT: PASS')

## 3. Bootstrap + System Check

In [ ]:
!python training/kaggle/scripts/bootstrap.py --skip-install

In [ ]:
!python training/kaggle/scripts/system_check.py

## 4. AMP Stability Check

In [ ]:
!python training/kaggle/scripts/amp_stability_test.py

## 5. GPU Smoke Test

In [ ]:
!python training/kaggle/scripts/gpu_smoke_test.py

## 6. Full Pipeline — Frozen Corpus + Training

This is the main training cell. It:
1. Loads the frozen R5.2.7 corpus via `FrozenCorpusLoader`
2. Validates the manifest (checksum, row count, class counts, split)
3. Prints the R5.4 data contract
4. Builds `AgriculturalObservation` objects (STAM imagery resolution)
5. Verifies the data contract at runtime
6. Trains the CropFusion multimodal model for 30 epochs
7. Saves best.pt + latest.pt checkpoints

In [ ]:
import os

lightweight = os.environ.get('R5_4_LIGHTWEIGHT') == '1'
epochs_env = os.environ.get('R5_4_EPOCHS', '').strip()
print('MODE:', 'LIGHTWEIGHT GATE (1 epoch)' if lightweight else ('EPOCHS=' + epochs_env if epochs_env else 'FULL R5.4 TRAINING (28-30 epochs)'))

# R5.4 pre-flight: numeric probe guards FP16/FP32 validation integrity under
# the new config (macro-F1 monitor, warmup-cosine, best-restore). Cheap;
# skips training if imagery is unavailable and exits nonzero on failure.
!python training/kaggle/scripts/validation_numerics_probe.py --frozen-crop-csv govt_crop_matched_v1/crop_supervised_v1.csv --frozen-manifest training_manifests/crop_supervised_v1_manifest.json --training-config training/config/training.yaml --model-config training/config/model.yaml --preprocessing-config training/config/preprocessing.yaml

cmd = [sys.executable or 'python', 'training/kaggle/scripts/run_pipeline.py',
       '--frozen-crop-csv', 'govt_crop_matched_v1/crop_supervised_v1.csv',
       '--frozen-manifest', 'training_manifests/crop_supervised_v1_manifest.json']
if lightweight:
    cmd.append('--lightweight')
if epochs_env:
    cmd.extend(['--epochs', epochs_env])
print('[cell16] running:', ' '.join(cmd))
p = subprocess.run(cmd, cwd=REPO_ROOT)
print('run_pipeline_exit=', p.returncode)
if p.returncode != 0:
    print('[FAIL] run_pipeline terminated with exit', p.returncode)
    raise SystemExit(p.returncode)


## 7. Multimodal Tensor Verification

In [ ]:
!python training/kaggle/scripts/verify_multimodal_tensors.py --corpus training/kaggle/outputs/reports/frozen_corpus.json

## 8. Split Composition Verification

In [ ]:
!python training/kaggle/scripts/verify_split_composition.py --corpus training/kaggle/outputs/reports/frozen_corpus.json

## 9. Checkpoint Report

In [ ]:
import json
from pathlib import Path

report_path = REPO_ROOT / 'training/kaggle/outputs/reports/pipeline.json'
info = {'found': False}

if report_path.exists():
    pipeline = json.loads(report_path.read_text())
    training = pipeline.get('training', {})
    if training.get('status') == 'completed' and training.get('run_dir'):
        run_dir = Path(training['run_dir'])
        ckpt_dir = run_dir / 'checkpoints'
        candidates = sorted(
            [p for p in ckpt_dir.rglob('*.pt')] if ckpt_dir.exists() else [],
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        if candidates:
            latest = candidates[0]
            from training.kaggle.config import load_paths_config, WorkspaceLayout
            from training.kaggle.workspace import WorkspaceManager
            paths = load_paths_config()
            layout = WorkspaceLayout.resolve(paths, repo_root=REPO_ROOT)
            workspace = WorkspaceManager(layout)
            workspace.create()
            report = training.get('report', {})
            entry = workspace.checkpoints.register(
                run_name=report.get('run_name') or run_dir.name,
                stage='best',
                metrics=report.get('evaluation', {}) or report.get('training', {}),
                path=str(latest),
                resume=True,
            )
            info = {
                'found': True,
                'path': str(latest),
                'run_dir': str(run_dir),
                'registered': entry,
            }
            print('latest checkpoint:', latest)
        else:
            print('no checkpoint files under', ckpt_dir)
    else:
        print('training status:', training.get('status'), '-', training.get('reason', ''))
else:
    print('pipeline report missing:', report_path)

out = REPO_ROOT / 'training/kaggle/outputs/reports/checkpoint.json'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(info, indent=2), encoding='utf-8')
print('checkpoint report ->', out)

## 10. Package Sources for Export

In [ ]:
!python training/kaggle/scripts/package_sources.py; echo "package_sources_exit=$?"

## 11. Summary

In [ ]:
import json
from pathlib import Path

pipeline_path = REPO_ROOT / 'training/kaggle/outputs/reports/pipeline.json'
if pipeline_path.exists():
    p = json.loads(pipeline_path.read_text())
    print('=' * 50)
    print('  R5.4 TRAINING SUMMARY')
    print('=' * 50)
    corpus = p.get('corpus', {})
    training = p.get('training', {})
    print(f'  Corpus: {corpus.get("type", "unknown")}')
    print(f'  Total: {corpus.get("total", "?")}')
    print(f'  Train: {corpus.get("train", "?")}')
    print(f'  Val:   {corpus.get("val", "?")}')
    print(f'  Test:  {corpus.get("test", "?")}')
    print(f'  Status: {training.get("status", "unknown")}')
    report = training.get('report', {})
    if report:
        print(f'  Run dir: {training.get("run_dir", "?")}')
        eval_result = report.get('evaluation', {})
        if eval_result:
            metrics = eval_result.get('metrics', {})
            print(f'  Test accuracy: {metrics.get("crop/accuracy", "?")}')
            print(f'  Test macro F1: {metrics.get("crop/macro_f1", "?")}')
            print(f'  Test weighted F1: {metrics.get("crop/weighted_f1", "?")}')
    print('=' * 50)
else:
    print('No pipeline report found')
